In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Predicting bycycle traffic across Seattle's Fremont bridge

In [2]:
fremont = pd.read_csv('fremont.csv',index_col='Date',parse_dates=True)
fremont

,"Fremont Bridge Sidewalks, south of N 34th St","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk"
Date,,,
2022-08-01 00:00:00,23.0,7.0,16.0
2022-08-01 01:00:00,12.0,5.0,7.0
2022-08-01 02:00:00,3.0,0.0,3.0
2022-08-01 03:00:00,5.0,2.0,3.0
2022-08-01 04:00:00,10.0,2.0,8.0
...,...,...,...
2023-08-31 19:00:00,224.0,72.0,152.0
2023-08-31 20:00:00,142.0,59.0,83.0
2023-08-31 21:00:00,67.0,35.0,32.0


In [3]:
# compute daily traffic
fremont = fremont.resample('d').sum()
fremont

,"Fremont Bridge Sidewalks, south of N 34th St","Fremont Bridge Sidewalks, south of N 34th St Cyclist East Sidewalk","Fremont Bridge Sidewalks, south of N 34th St Cyclist West Sidewalk"
Date,,,
2012-10-03,3521.0,1760.0,1761.0
2012-10-04,3475.0,1708.0,1767.0
2012-10-05,3148.0,1558.0,1590.0
2012-10-06,2006.0,1080.0,926.0
2012-10-07,2142.0,1191.0,951.0
...,...,...,...
2023-08-27,2169.0,936.0,1233.0
2023-08-28,3027.0,1026.0,2001.0
2023-08-29,2767.0,842.0,1925.0


In [4]:
# plot daily traffic
fremont['Fremont Bridge Total'].plot(figsize=(12,5))

KeyError: 'Fremont Bridge Total'

In [ ]:
# day of the week/month/year/covid
fremont['day_of_week'] = fremont.index.dayofweek
fremont['month'] = fremont.index.month
fremont['year'] = fremont.index.year
fremont['covid'] = 0
fremont.loc['03-01-2020':,'covid'] = 1
fremont

In [ ]:
# holidays
from pandas.tseries.holiday import USFederalHolidayCalendar
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays('01-01-2013','09-30-2022')
fremont['holidays'] = pd.Series(1,index=holidays,name='holidays')
fremont.fillna(0,inplace=True)
fremont

In [ ]:
fremont.holidays.value_counts()

In [ ]:
# hours of daylight
def get_hoursdaylight(date):
    axis = np.radians(23.44) # tilt of Earth's axis
    latitude = np.radians(47.61) # Seattle's latidude 
    days = (date-pd.to_datetime('2000-12-31')).days
    m = (1 - np.tan(latitude)*np.tan(axis*np.cos(days*2*np.pi/365.25)))
    return 24*np.degrees(np.arccos(1-m))/180
fremont['hours_daylight'] = fremont.index.map(get_hoursdaylight)

In [ ]:
fremont['hours_daylight'] = fremont.index.map(get_hoursdaylight)

In [ ]:
fremont.hours_daylight.plot(figsize=(12,3))

In [ ]:
# weather features
weather = pd.read_csv('weather.csv',index_col='DATE',parse_dates=True)
# PRCP (precipitation), TAVG (average temperature), SNOW (snowfall), AWND (average wind speed)
weather[['PRCP','TAVG','SNOW','AWND']].isna().sum()

In [ ]:
# TAVG column has some missing values 
weather.TAVG.plot(figsize=(12,5))

In [ ]:
# fix TAVG column
weather.TAVG.fillna(0.5*(weather.TMAX+weather.TMIN),inplace=True)

In [ ]:
# plot precipitation data
weather.PRCP.plot()

In [ ]:
# plot snowfall data
weather.SNOW.plot()

In [ ]:
# plot wind data
weather.AWND.plot()

In [ ]:
fremont = fremont.join(weather[['PRCP','TAVG','SNOW','AWND']]).loc['2013-01-01':]
fremont

In [ ]:
# add air quality index
aqi = pd.read_csv('Seattle_air_quality_index.csv',index_col='Date',parse_dates=True)
fremont['aqi'] = aqi

In [ ]:
# fix aqi missing values
fremont.fillna(method='ffill',inplace=True)

In [ ]:
# 2013-2020
fremont2013_2020 = fremont[fremont.index.year<2021].copy()
# 2020-2022
fremont2021_2022 = fremont[fremont.index.year>=2021].copy()

In [ ]:
# target vector / feature matrix
y2013_2020 = fremont2013_2020['Fremont Bridge Total']
X2013_2020 = fremont2013_2020[['day_of_week','covid','holidays','hours_daylight','PRCP','TAVG','SNOW','AWND','year','aqi']]

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, PolynomialFeatures

In [ ]:
# regression pipeline

num_features = ['hours_daylight','PRCP','TAVG','SNOW','AWND','year','aqi']
cat_features = ['day_of_week','covid','holidays']


feature_processor = ColumnTransformer(transformers=[
    ('encoder',OneHotEncoder(),cat_features),
],remainder='passthrough')

pipe = Pipeline(steps=[
    ('feature_processor',feature_processor),
    ('poly_features',PolynomialFeatures(degree=1,add_bias=False)),
    ('regression',LinearRegression())
])

In [ ]:
pipe.fit(X2013_2020,y2013_2020)
fremont2013_2020['predicted'] = pipe.predict(X2013_2020)

In [ ]:
# daily traffic: actual and predicted
fremont2013_2020[['Fremont Bridge Total','predicted']].plot(figsize=(12,5),alpha=0.7)

In [ ]:
fremont2013_2020.plot.scatter(x='Fremont Bridge Total',y='predicted')
plt.plot([0,6000],[0,6000],'r--')
plt.title('daily traffic')

In [ ]:
# monthly traffic: actual and predicted
fremont2013_2020[['Fremont Bridge Total','predicted']].resample('m').sum().plot(figsize=(12,5))

In [ ]:
fremont2013_2020.resample('m').sum().plot.scatter(x='Fremont Bridge Total',y='predicted')
plt.plot([40_000,140_000],[40_000,140_000],'r--')
plt.title('monthly traffic')

In [ ]:
# yearly traffic: actual and predicted
fremont2013_2020[['Fremont Bridge Total','predicted']].resample('y').sum().plot(figsize=(12,5))

## Data Science Fiction: 2021-2022 without COVID

In [ ]:
# set covid feature to zero
fremont2021_2022.covid = 0

In [ ]:
X2021_2022 = fremont2021_2022[['day_of_week','covid','holidays',
                              'hours_daylight','year','PRCP','TAVG','SNOW','AWND','aqi']]
y2021_2022 = fremont2021_2022['Fremont Bridge Total']

In [ ]:
fremont2021_2022['prediction'] = pipe.predict(X2021_2022)

In [ ]:
fremont2021_2022[['Fremont Bridge Total','prediction']].plot(figsize=(15,5))

## Regularization + GridSearch

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# regression pipeline
pipe = Pipeline(steps=[
    ('feature_processor',feature_processor),
    ('scaler', MinMaxScaler()),
    ('poly_features',PolynomialFeatures(degree=1)),
    ('regressor',Ridge()) #  Rige, Lasso or LinearRegression
])

In [ ]:
param_dic = {'poly_features__degree':[1,2],
             'regressor__alpha':[1e-5,1e-4,1e-3,1e-2,1e-1,1,10,100,1000]}

In [ ]:
grid = GridSearchCV(pipe,
             param_dic,
             scoring='neg_mean_squared_error' ,
             cv=10,
             n_jobs=-1,
             verbose=1)

In [ ]:
grid.fit(X2013_2020,y2013_2020)

In [ ]:
grid.best_params_

In [ ]:
best_pipe = grid.best_estimator_

In [ ]:
fremont2013_2020['prediction'] = best_pipe.predict(X2013_2020)

In [ ]:
# daily traffic: actual and predicted
fremont2013_2020[['Fremont Bridge Total','predicted']].plot(figsize=(15,5),alpha=0.7)

In [ ]:
fremont2013_2020.plot.scatter(x='Fremont Bridge Total',y='predicted')
plt.plot([0,6000],[0,6000],'r--')
plt.title('daily traffic')

## Model's coefficients (no polynomial features)

In [ ]:
# pipeline
pipe = Pipeline(steps=[
    ('feature_processor',feature_processor),
    ('poly_features',PolynomialFeatures(degree=1,include_bias=False)),
    ('regressor', LinearRegression()) # or Ridge or Lasso
])
pipe.fit(X2013_2020,y2013_2020)

In [ ]:
cat_features

In [ ]:
# encoded feature names
encoded_feature_names = pipe['feature_processor'].named_transformers_['encoder'].get_feature_names_out(cat_features)
list(encoded_feature_names)

In [ ]:
feature_names = list(encoded_feature_names)+num_features
feature_names

In [ ]:
# coefficients
coefficients = pipe['regressor'].coef_
coefficients

In [ ]:
len(coefficients)

In [ ]:
len(feature_names)

In [ ]:
# put coefficients into a dataframe
coeff_df = pd.DataFrame(data=coefficients,index=feature_names,columns=['coefficient'])
coeff_df

In [ ]:
coeff_df.plot.barh(figsize=(10,10))

## Model's coefficients (no polynomial features)

In [ ]:
# pipeline
pipe = Pipeline(steps=[
    ('feature_processor',feature_processor),
    ('poly_features',PolynomialFeatures(degree=2,include_bias=False)),
    ('regressor', Ridge(alpha=1)) # Ridge, Lasso or Linear Regression
])
pipe.fit(X2013_2020,y2013_2020)

In [ ]:
poly_feature_names = pipe['poly_features'].get_feature_names_out(feature_names)
list(poly_feature_names)

In [ ]:
len(poly_feature_names)

In [ ]:
coefficients = pipe['regressor'].coef_
len(coefficients)

In [ ]:
coeff_df = pd.DataFrame(data=coefficients, index=poly_feature_names, columns=['coefficient'])
coeff_df

In [ ]:
sorted_coeff_df = coeff_df.sort_values(by='coefficient', ascending=False)
sorted_coeff_df

In [ ]:
# top 5 positive coefficients
sorted_coeff_df.head(5).plot.barh(figsize=(10,5))

In [ ]:
# top 5 negative coefficients
sorted_coeff_df.tail(5).plot.barh(figsize=(10,5))